In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import random
from scipy import stats

In [30]:
# Set seed for reproducibility so you get the exact same data every time
np.random.seed(42)
random.seed(42)

# 1. Define Business Categories
n_rows = 100
regions = ['North America', 'Europe', 'Asia-Pacific', 'Latin America']
products = ['Laptops', 'Smartphones', 'Audio', 'Wearables']
tiers = ['Standard', 'Premium', 'VIP']
accessories_pool = ['Case', 'Screen Protector', 'Extra Charger', 'Wireless Mouse', 'Headphones']

# 2. Generate Base Data
data = {
    'Customer_ID': [f"CUST-{str(i).zfill(4)}" for i in range(1, n_rows + 1)],
    'Region': np.random.choice(regions, n_rows),
    'Product_Category': np.random.choice(products, n_rows),
    'Subscription_Tier': np.random.choice(tiers, n_rows, p=[0.6, 0.3, 0.1]),
    'Customer_Rating': np.random.randint(1, 6, n_rows),
    'Discount_Used': np.random.choice(['Yes', 'No'], n_rows, p=[0.4, 0.6]),

    # Wide Panel Data: Quarterly Sales Revenue (in USD)
    'Sales_Q1': np.round(np.random.normal(500, 100, n_rows), 2),
    'Sales_Q2': np.round(np.random.normal(550, 120, n_rows), 2),
    'Sales_Q3': np.round(np.random.normal(480, 90, n_rows), 2),
    'Sales_Q4': np.round(np.random.normal(700, 150, n_rows), 2), # Holiday spike

    # Wide Panel Data: Quarterly Support Tickets logged by the customer
    'Tickets_Q1': np.random.poisson(1, n_rows),
    'Tickets_Q2': np.random.poisson(1.5, n_rows),
    'Tickets_Q3': np.random.poisson(0.5, n_rows),
    'Tickets_Q4': np.random.poisson(2, n_rows),
}

# 3. Create the DataFrame
df = pd.DataFrame(data)
print(df.head())

  Customer_ID         Region Product_Category Subscription_Tier  \
0   CUST-0001   Asia-Pacific            Audio          Standard   
1   CUST-0002  Latin America      Smartphones           Premium   
2   CUST-0003  North America      Smartphones          Standard   
3   CUST-0004   Asia-Pacific        Wearables          Standard   
4   CUST-0005   Asia-Pacific      Smartphones               VIP   

   Customer_Rating Discount_Used  Sales_Q1  Sales_Q2  Sales_Q3  Sales_Q4  \
0                4           Yes    278.32    512.83    401.59    625.47   
1                3            No    638.96    409.62    459.59    758.18   
2                1            No    687.04    537.04    410.60    798.81   
3                4            No    668.80    733.25    531.86    563.63   
4                4           Yes    438.48    476.11    587.06    946.31   

   Tickets_Q1  Tickets_Q2  Tickets_Q3  Tickets_Q4  
0           0           2           1           2  
1           1           1           

# Data Transformations

## Explode

In [31]:
data = {
    'Customer_ID': ['CUST-001', 'CUST-002', 'CUST-003', 'CUST-004'],
    'Main_Purchase': ['Smartphone', 'Laptop', 'Audio', 'Wearable'],
    'Accessories_Bought': [
        ['Case', 'Screen Protector'],           # Customer 1 bought 2 items
        ['Wireless Mouse'],                     # Customer 2 bought 1 item
        [],                                     # Customer 3 bought ZERO items (empty list)
        ['Headphones', 'Charger', 'Case']       # Customer 4 bought 3 items
    ]
}

df_lists = pd.DataFrame(data)
print(df_lists)

  Customer_ID Main_Purchase           Accessories_Bought
0    CUST-001    Smartphone     [Case, Screen Protector]
1    CUST-002        Laptop             [Wireless Mouse]
2    CUST-003         Audio                           []
3    CUST-004      Wearable  [Headphones, Charger, Case]


In [32]:
df_explode = df_lists.explode('Accessories_Bought')
print(df_explode)

  Customer_ID Main_Purchase Accessories_Bought
0    CUST-001    Smartphone               Case
0    CUST-001    Smartphone   Screen Protector
1    CUST-002        Laptop     Wireless Mouse
2    CUST-003         Audio                NaN
3    CUST-004      Wearable         Headphones
3    CUST-004      Wearable            Charger
3    CUST-004      Wearable               Case


## Pivot, Melt and WideToLong

In [5]:
prod_region = pd.pivot_table(df, index='Product_Category', columns='Region', values='Sales_Q1', aggfunc='sum')
print(prod_region)

Region            Asia-Pacific   Europe  Latin America  North America
Product_Category                                                     
Audio                  2477.45  2310.82        6911.02        2525.35
Laptops                5321.41  3727.52        1565.06        2233.60
Smartphones            1980.66  3846.53        2127.37        2013.94
Wearables              2163.43  2847.27        3147.22        3281.21


In [6]:
prod_region.reset_index(inplace=True)
print(prod_region.columns)
print(prod_region.index)
print(prod_region)
# prod_region.drop(columns=['index'], inplace=True)
# print(prod_region.columns)

Index(['Product_Category', 'Asia-Pacific', 'Europe', 'Latin America',
       'North America'],
      dtype='str', name='Region')
RangeIndex(start=0, stop=4, step=1)
Region Product_Category  Asia-Pacific   Europe  Latin America  North America
0                 Audio       2477.45  2310.82        6911.02        2525.35
1               Laptops       5321.41  3727.52        1565.06        2233.60
2           Smartphones       1980.66  3846.53        2127.37        2013.94
3             Wearables       2163.43  2847.27        3147.22        3281.21


In [7]:
melt_prod = pd.melt(prod_region, id_vars='Product_Category', value_vars='North America', var_name='Region', value_name='NA Sales_Q1')

print(melt_prod)

  Product_Category         Region  NA Sales_Q1
0            Audio  North America      2525.35
1          Laptops  North America      2233.60
2      Smartphones  North America      2013.94
3        Wearables  North America      3281.21


In [8]:
wl_prod = pd.wide_to_long(df, ['Sales_Q', 'Tickets_Q'], i='Customer_ID', j='Quarter', sep='')
print(wl_prod)

                     Customer_Rating Discount_Used Product_Category  \
Customer_ID Quarter                                                   
CUST-0001   1                      4           Yes            Audio   
CUST-0002   1                      3            No      Smartphones   
CUST-0003   1                      1            No      Smartphones   
CUST-0004   1                      4            No        Wearables   
CUST-0005   1                      4           Yes      Smartphones   
...                              ...           ...              ...   
CUST-0096   4                      4           Yes      Smartphones   
CUST-0097   4                      2            No      Smartphones   
CUST-0098   4                      2           Yes        Wearables   
CUST-0099   4                      3            No          Laptops   
CUST-0100   4                      1            No            Audio   

                            Region Subscription_Tier  Sales_Q  Tickets_Q  
C

## Assign

In [9]:
df = df.assign(TotalSales=df['Sales_Q1'] + df['Sales_Q2'] + df['Sales_Q3'] + df['Sales_Q4'])
print(df.head())


  Customer_ID         Region Product_Category Subscription_Tier  \
0   CUST-0001   Asia-Pacific            Audio          Standard   
1   CUST-0002  Latin America      Smartphones           Premium   
2   CUST-0003  North America      Smartphones          Standard   
3   CUST-0004   Asia-Pacific        Wearables          Standard   
4   CUST-0005   Asia-Pacific      Smartphones               VIP   

   Customer_Rating Discount_Used  Sales_Q1  Sales_Q2  Sales_Q3  Sales_Q4  \
0                4           Yes    278.32    512.83    401.59    625.47   
1                3            No    638.96    409.62    459.59    758.18   
2                1            No    687.04    537.04    410.60    798.81   
3                4            No    668.80    733.25    531.86    563.63   
4                4           Yes    438.48    476.11    587.06    946.31   

   Tickets_Q1  Tickets_Q2  Tickets_Q3  Tickets_Q4  TotalSales  
0           0           2           1           2     1818.21  
1           

In [10]:
df = df.assign(TotalTickets=lambda x: x['Tickets_Q1'] + x['Tickets_Q2'] + x['Tickets_Q3'] + x['Tickets_Q4'])
print(df.head())

  Customer_ID         Region Product_Category Subscription_Tier  \
0   CUST-0001   Asia-Pacific            Audio          Standard   
1   CUST-0002  Latin America      Smartphones           Premium   
2   CUST-0003  North America      Smartphones          Standard   
3   CUST-0004   Asia-Pacific        Wearables          Standard   
4   CUST-0005   Asia-Pacific      Smartphones               VIP   

   Customer_Rating Discount_Used  Sales_Q1  Sales_Q2  Sales_Q3  Sales_Q4  \
0                4           Yes    278.32    512.83    401.59    625.47   
1                3            No    638.96    409.62    459.59    758.18   
2                1            No    687.04    537.04    410.60    798.81   
3                4            No    668.80    733.25    531.86    563.63   
4                4           Yes    438.48    476.11    587.06    946.31   

   Tickets_Q1  Tickets_Q2  Tickets_Q3  Tickets_Q4  TotalSales  TotalTickets  
0           0           2           1           2     1818.21 

## Crosstab

In [11]:
df_cross_ex1 = pd.crosstab(index=df['Region'], columns=df['Product_Category'], rownames=['Continents'], colnames=['Products'])
print(df_cross_ex1)

Products       Audio  Laptops  Smartphones  Wearables
Continents                                           
Asia-Pacific       5       11            4          4
Europe             5        7            8          6
Latin America     15        4            4          7
North America      5        4            4          7


In [12]:
df_cross_ex2 = pd.crosstab(index=df['Region'], columns=df['Product_Category'], rownames=['Continents'], colnames=['Products'], margins=True, margins_name='MeansAll', values=df['Sales_Q1'], aggfunc='mean')
print(df_cross_ex2.round(2))

Products        Audio  Laptops  Smartphones  Wearables  MeansAll
Continents                                                      
Asia-Pacific   495.49   483.76       495.16     540.86    497.62
Europe         462.16   532.50       480.82     474.54    489.70
Latin America  460.73   391.26       531.84     449.60    458.36
North America  505.07   558.40       503.48     468.74    502.71
MeansAll       474.15   494.14       498.42     476.63    484.80


Comparing with pivot_table

In [13]:
df_pivot_example = pd.pivot_table(df, index='Region', columns='Product_Category', values=['Sales_Q1', 'Sales_Q2', 'Sales_Q3', 'Sales_Q4'], aggfunc='mean', margins=True, margins_name='MeanAll')
print(df_pivot_example.round(2))

                 Sales_Q1                                       Sales_Q2  \
Product_Category    Audio Laptops Smartphones Wearables MeanAll    Audio   
Region                                                                     
Asia-Pacific       495.49  483.76      495.16    540.86  497.62   489.51   
Europe             462.16  532.50      480.82    474.54  489.70   581.43   
Latin America      460.73  391.26      531.84    449.60  458.36   531.09   
North America      505.07  558.40      503.48    468.74  502.71   586.73   
MeanAll            474.15  494.14      498.42    476.63  484.80   541.82   

                                                       Sales_Q3          \
Product_Category Laptops Smartphones Wearables MeanAll    Audio Laptops   
Region                                                                    
Asia-Pacific      508.74      539.74    623.91  529.09   433.58  505.77   
Europe            570.59      594.51    530.45  570.77   498.28  443.86   
Latin America   

## Cut

In [14]:
bin_edges = [0, 1500, 2000, 2500, 3000, np.inf]
bin_labels = ['XS', 'S', 'M', 'L', 'XL']

df['SalesBin'] = pd.cut(df.TotalSales, bins=bin_edges, labels=bin_labels, include_lowest=True)
# print(df_cut)
df.SalesBin.value_counts()

SalesBin
M     75
S     15
L      8
XS     1
XL     1
Name: count, dtype: int64

## Get Dummies

In [15]:
df = pd.get_dummies(df, columns=['Region'], dtype=int)
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype   
---  ------                --------------  -----   
 0   Customer_ID           100 non-null    str     
 1   Product_Category      100 non-null    str     
 2   Subscription_Tier     100 non-null    str     
 3   Customer_Rating       100 non-null    int64   
 4   Discount_Used         100 non-null    str     
 5   Sales_Q1              100 non-null    float64 
 6   Sales_Q2              100 non-null    float64 
 7   Sales_Q3              100 non-null    float64 
 8   Sales_Q4              100 non-null    float64 
 9   Tickets_Q1            100 non-null    int64   
 10  Tickets_Q2            100 non-null    int64   
 11  Tickets_Q3            100 non-null    int64   
 12  Tickets_Q4            100 non-null    int64   
 13  TotalSales            100 non-null    float64 
 14  TotalTickets          100 non-null    int64   
 15  SalesBin          

## Stack, unstack, set_index and reset_index

In [87]:
data_v1 = {
    'Store': ['North', 'South'],
    'Product': ['Apples', 'Bananas'],
    'Q1': [100, 150],
    'Q2': [120, 170]
}
df_v1 = pd.DataFrame(data_v1)
print(df_v1)

   Store  Product   Q1   Q2
0  North   Apples  100  120
1  South  Bananas  150  170


In [88]:
# Set index

df_v1.set_index(['Store', 'Product'], inplace=True)
print(df_v1)

                Q1   Q2
Store Product          
North Apples   100  120
South Bananas  150  170


In [89]:
# Reset_index

df_v1.reset_index(inplace=True)
print(df_v1)

   Store  Product   Q1   Q2
0  North   Apples  100  120
1  South  Bananas  150  170


In [92]:
# Stack

df_v2 = df_v1.stack(level=0)
print(df_v2)
print(type(df_v2))

0  Store        North
   Product     Apples
   Q1             100
   Q2             120
1  Store        South
   Product    Bananas
   Q1             150
   Q2             170
dtype: object
<class 'pandas.Series'>


In [97]:
# Unstack

df_v3 = df_v2.unstack(level=1)
print(df_v3)
print(type(df_v3))
print(df_v3.index)
print(df_v3.columns)

   Store  Product   Q1   Q2
0  North   Apples  100  120
1  South  Bananas  150  170
<class 'pandas.DataFrame'>
RangeIndex(start=0, stop=2, step=1)
Index(['Store', 'Product', 'Q1', 'Q2'], dtype='str')


# DRAFT

In [85]:
del(df_v1)



In [17]:
example = np.random.randint(1, 100, size=100)
example_series = pd.Series(example)
print(example_series)

0      9
1     41
2      1
3      2
4     60
      ..
95    98
96    11
97    10
98    15
99    17
Length: 100, dtype: int64


In [18]:
example_series[0:5]

0     9
1    41
2     1
3     2
4    60
dtype: int64

In [ ]:
df.index

In [ ]:
# Poisson distribution

poisson = np.random.poisson(20, 1000)
poisson_data = pd.Series(poisson)
g = sns.histplot(poisson_data, bins=10, kde=True)
g.set_title('Poisson Distribution Histogram')

print(poisson_data.describe())
print(poisson_data.head(5))

In [ ]:
# Checking normality
shapiro = stats.shapiro(poisson_data)
print(f'Shapiro-Wilk Test: {shapiro.pvalue}')

anderson = stats.anderson(poisson_data, dist='norm', method='interpolate')
print(f'Anderson-Darling Test: {anderson.pvalue}')

# Both statistical tests are showing that the H0 is rejected, meaning the data is not Normal
# this validates the fact that the data was created with

In [ ]:
example = pd.DataFrame(
    {
        "A": {0: "a", 1: "b", 2: "c"},
        "B": {0: 1, 1: 3, 2: 5},
        "C": {0: 2, 1: 4, 2: 6},
    }
)

example

In [ ]:
example.columns

In [ ]:
example.index

In [ ]:
prod_region.columns

In [ ]:
prod_region.columns

In [ ]:
prod_region.index

In [ ]:
prod_region.index

In [ ]:
print(prod_region)

In [ ]:
prod_region.Region